# Experiment 01 — Random Forest Parity on Resampled Watch Sequences

## Purpose and decision rule

This experiment asks whether the shared 20 Hz paired-watch representation
preserves the information captured by the original Random Forest feature-fusion
baseline.

The reference model used 56 whole-window statistical features extracted from
the original watch accelerometer and gyroscope samples. Its mean subject-fold
macro-F1 was **0.6907 ± 0.0480**.

This experiment uses:

- the same 25,585 paired development examples;
- the same 56 feature definitions;
- the same five subject-wise folds;
- the same Random Forest configuration;
- development data only.

The 20 Hz representation passes the practical parity gate if its mean
validation macro-F1 is at least **0.67**. This threshold represents no more than
an approximately 0.02 absolute decrease from the reference result. It is a
development decision rule, not a formal statistical-equivalence test.

No test artifact is loaded in this notebook.

## 1. Load the development artifact

Only the paired-watch training artifact is loaded. The sequence arrays contain
unnormalized accelerometer and gyroscope signals with channel order `x, y, z`.
The metadata supplies the activity labels, subjects, and fixed fold assignments.

Coverage masks are not used as Random Forest features in this parity experiment.
All 100 positions of the fixed representation are included.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        candidate
        for candidate in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (
            candidate
            / "modified"
            / "watch_pair_20hz"
            / "preprocessing_meta.json"
        ).exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the activity_project root."
    )

ARTIFACT_DIR = (
    PROJECT_ROOT
    / "modified"
    / "watch_pair_20hz"
)

TRAIN_SEQUENCE_PATH = (
    ARTIFACT_DIR / "train_sequences.npz"
)
TRAIN_METADATA_PATH = (
    ARTIFACT_DIR / "train_metadata.parquet"
)
PREPROCESSING_META_PATH = (
    ARTIFACT_DIR / "preprocessing_meta.json"
)

preprocessing_meta = json.loads(
    PREPROCESSING_META_PATH.read_text(
        encoding="utf-8"
    )
)

with np.load(TRAIN_SEQUENCE_PATH) as artifact:
    train_accel = artifact["accel"]
    train_gyro = artifact["gyro"]
    target_offsets_s = artifact[
        "target_offsets_s"
    ]

train_metadata = pd.read_parquet(
    TRAIN_METADATA_PATH
)

expected_shape = tuple(
    preprocessing_meta["outputs"][
        "train_sequences"
    ]["accel_shape"]
)

assert train_accel.shape == expected_shape
assert train_gyro.shape == expected_shape
assert len(train_metadata) == expected_shape[0]

assert np.array_equal(
    train_metadata["array_index"].to_numpy(),
    np.arange(len(train_metadata)),
)

assert set(
    train_metadata["fold"].astype(int)
) == {1, 2, 3, 4, 5}

artifact_overview = pd.Series({
    "Examples": len(train_metadata),
    "Accelerometer shape": train_accel.shape,
    "Gyroscope shape": train_gyro.shape,
    "Signal dtype": str(train_accel.dtype),
    "Target rate (Hz)": (
        preprocessing_meta[
            "resampling"
        ]["target_rate_hz"]
    ),
    "Subjects": train_metadata[
        "user_id"
    ].nunique(),
    "Activities": train_metadata[
        "activity_code"
    ].nunique(),
    "Development folds": train_metadata[
        "fold"
    ].nunique(),
})

display(artifact_overview.to_frame("value"))
display(
    train_metadata[
        [
            "array_index",
            "pair_id",
            "user_id",
            "activity_code",
            "fold",
        ]
    ].head()
)

print("Development artifact loaded.")

,value
Examples,25585
Accelerometer shape,"(25585, 100, 3)"
Gyroscope shape,"(25585, 100, 3)"
Signal dtype,float32
Target rate (Hz),20
Subjects,41
Activities,18
Development folds,5


,array_index,pair_id,user_id,activity_code,fold
0,0,1600_watch_pair_A_0000,1600,A,3
1,1,1600_watch_pair_A_0001,1600,A,3
2,2,1600_watch_pair_A_0002,1600,A,3
3,3,1600_watch_pair_A_0003,1600,A,3
4,4,1600_watch_pair_A_0004,1600,A,3


Development artifact loaded.


## 2. Recreate the baseline statistical features

For each sensor, magnitude is calculated at every timestep as
`√(x² + y² + z²)`. Seven statistics are then extracted from each of `x`, `y`,
`z`, and magnitude:

- mean;
- sample standard deviation (`ddof=1`, matching pandas);
- minimum and maximum;
- median;
- interquartile range;
- RMS.

This produces 28 features per sensor and 56 fused features. Feature extraction
is deterministic and learns no population parameters.

In [2]:
SIGNAL_NAMES = [
    "x",
    "y",
    "z",
    "magnitude",
]

STATISTIC_NAMES = [
    "mean",
    "std",
    "min",
    "max",
    "median",
    "iqr",
    "rms",
]


def extract_sensor_features(
    sequences,
    sensor_prefix,
):
    xyz = sequences.astype(
        np.float64,
        copy=False,
    )

    magnitude = np.sqrt(
        np.sum(
            np.square(xyz),
            axis=2,
            keepdims=True,
        )
    )

    signals = np.concatenate(
        [xyz, magnitude],
        axis=2,
    )

    first_quartile, third_quartile = (
        np.quantile(
            signals,
            [0.25, 0.75],
            axis=1,
            method="linear",
        )
    )

    statistic_values = {
        "mean": signals.mean(axis=1),
        "std": signals.std(
            axis=1,
            ddof=1,
        ),
        "min": signals.min(axis=1),
        "max": signals.max(axis=1),
        "median": np.median(
            signals,
            axis=1,
        ),
        "iqr": (
            third_quartile
            - first_quartile
        ),
        "rms": np.sqrt(
            np.mean(
                np.square(signals),
                axis=1,
            )
        ),
    }

    feature_names = [
        (
            f"{sensor_prefix}_"
            f"{signal_name}_"
            f"{statistic_name}"
        )
        for signal_name in SIGNAL_NAMES
        for statistic_name in STATISTIC_NAMES
    ]

    feature_matrix = np.column_stack([
        statistic_values[
            statistic_name
        ][:, signal_index]
        for signal_index, signal_name
        in enumerate(SIGNAL_NAMES)
        for statistic_name in STATISTIC_NAMES
    ])

    return feature_matrix, feature_names

In [3]:
(
    accel_feature_matrix,
    accel_feature_names,
) = extract_sensor_features(
    train_accel,
    "accel",
)

(
    gyro_feature_matrix,
    gyro_feature_names,
) = extract_sensor_features(
    train_gyro,
    "gyro",
)

X_parity = np.concatenate(
    [
        accel_feature_matrix,
        gyro_feature_matrix,
    ],
    axis=1,
)

feature_names = (
    accel_feature_names
    + gyro_feature_names
)

y = train_metadata[
    "activity_code"
].astype(str).to_numpy()

fold_assignments = train_metadata[
    "fold"
].astype(int).to_numpy()

subject_ids = train_metadata[
    "user_id"
].to_numpy()

assert X_parity.shape == (
    len(train_metadata),
    56,
)
assert len(feature_names) == 56
assert len(set(feature_names)) == 56
assert np.isfinite(X_parity).all()

feature_overview = pd.Series({
    "Examples": X_parity.shape[0],
    "Accelerometer features": (
        len(accel_feature_names)
    ),
    "Gyroscope features": (
        len(gyro_feature_names)
    ),
    "Total features": X_parity.shape[1],
    "Feature dtype": str(X_parity.dtype),
    "Finite values": bool(
        np.isfinite(X_parity).all()
    ),
})

display(feature_overview.to_frame("value"))

display(
    pd.DataFrame(
        X_parity[:3, :7],
        columns=feature_names[:7],
        index=train_metadata[
            "pair_id"
        ].iloc[:3],
    ).round(4)
)

print("Parity feature matrix constructed.")

,value
Examples,25585
Accelerometer features,28
Gyroscope features,28
Total features,56
Feature dtype,float64
Finite values,True


,accel_x_mean,accel_x_std,accel_x_min,accel_x_max,accel_x_median,accel_x_iqr,accel_x_rms
pair_id,,,,,,,
1600_watch_pair_A_0000,11.2953,4.2865,2.8306,19.6076,11.5972,7.1165,12.0737
1600_watch_pair_A_0001,12.8985,3.4095,6.3249,19.7611,12.6273,5.4275,13.3371
1600_watch_pair_A_0002,12.3451,3.1760,6.8095,19.5615,11.9479,5.0112,12.7431


Parity feature matrix constructed.


## 3. Fixed-fold Random Forest parity evaluation

The Random Forest is trained once for each fixed subject fold. Four folds form
the training population and the remaining subjects form validation data.

The model configuration is unchanged from the original fused baseline. No
normalization, class weighting, feature selection, or tuning is performed.
Every development example receives exactly one out-of-fold prediction.

In [4]:
from time import perf_counter

from sklearn.ensemble import (
    RandomForestClassifier,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
)


ACTIVITY_CODES = preprocessing_meta[
    "labels"
]["activity_codes_in_order"]

REFERENCE_MACRO_F1 = 0.6907
PARITY_THRESHOLD = 0.67


def calculate_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "balanced_accuracy": recall_score(
            y_true,
            y_pred,
            labels=ACTIVITY_CODES,
            average="macro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=ACTIVITY_CODES,
            average="macro",
            zero_division=0,
        ),
    }


oof_predictions = np.empty_like(y)
prediction_filled = np.zeros(
    len(y),
    dtype=bool,
)

fold_metric_rows = []
run_rows = []

for fold in sorted(
    np.unique(fold_assignments)
):
    training_mask = (
        fold_assignments != fold
    )
    validation_mask = (
        fold_assignments == fold
    )

    training_subjects = set(
        subject_ids[training_mask]
    )
    validation_subjects = set(
        subject_ids[validation_mask]
    )

    assert training_subjects.isdisjoint(
        validation_subjects
    )

    fold_model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    )

    start_time = perf_counter()

    fold_model.fit(
        X_parity[training_mask],
        y[training_mask],
    )

    fold_predictions = fold_model.predict(
        X_parity[validation_mask]
    )

    elapsed_seconds = (
        perf_counter() - start_time
    )

    oof_predictions[
        validation_mask
    ] = fold_predictions

    prediction_filled[
        validation_mask
    ] = True

    fold_metric_rows.append({
        "fold": int(fold),
        **calculate_metrics(
            y[validation_mask],
            fold_predictions,
        ),
    })

    run_rows.append({
        "fold": int(fold),
        "training_subjects": len(
            training_subjects
        ),
        "validation_subjects": len(
            validation_subjects
        ),
        "training_examples": int(
            training_mask.sum()
        ),
        "validation_examples": int(
            validation_mask.sum()
        ),
        "fit_predict_seconds": (
            elapsed_seconds
        ),
    })

    print(
        f"Fold {fold}: "
        f"{int(validation_mask.sum()):,} "
        f"predictions in "
        f"{elapsed_seconds:.2f} seconds"
    )

assert prediction_filled.all()

parity_oof_predictions = (
    train_metadata[
        [
            "array_index",
            "pair_id",
            "user_id",
            "activity_code",
            "fold",
        ]
    ]
    .rename(
        columns={
            "activity_code": "y_true",
        }
    )
    .copy()
)

parity_oof_predictions[
    "y_pred"
] = oof_predictions

assert len(
    parity_oof_predictions
) == len(train_metadata)

assert parity_oof_predictions[
    "pair_id"
].is_unique

parity_fold_metrics = pd.DataFrame(
    fold_metric_rows
)

parity_run_log = pd.DataFrame(
    run_rows
)

Fold 1: 5,670 predictions in 2.06 seconds
Fold 2: 4,970 predictions in 2.10 seconds
Fold 3: 4,970 predictions in 2.10 seconds
Fold 4: 4,970 predictions in 2.13 seconds
Fold 5: 5,005 predictions in 2.14 seconds


In [5]:
METRIC_COLUMNS = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
]

parity_fold_summary = (
    parity_fold_metrics[
        METRIC_COLUMNS
    ]
    .agg(["mean", "std"])
    .T
)

parity_macro_f1 = float(
    parity_fold_summary.loc[
        "macro_f1",
        "mean",
    ]
)

parity_passed = (
    parity_macro_f1
    >= PARITY_THRESHOLD
)

parity_decision = pd.Series({
    "Reference macro-F1": (
        REFERENCE_MACRO_F1
    ),
    "Resampled macro-F1": (
        parity_macro_f1
    ),
    "Difference from reference": (
        parity_macro_f1
        - REFERENCE_MACRO_F1
    ),
    "Parity threshold": (
        PARITY_THRESHOLD
    ),
    "Parity passed": parity_passed,
})

display(parity_run_log.round(3))
display(parity_fold_metrics.round(4))
display(parity_fold_summary.round(4))
display(
    parity_decision.to_frame("value")
)

print(
    "Parity decision:",
    "PASS" if parity_passed else "FAIL",
)

,fold,training_subjects,validation_subjects,training_examples,validation_examples,fit_predict_seconds
0,1,32,9,19915,5670,2.059
1,2,33,8,20615,4970,2.103
2,3,33,8,20615,4970,2.099
3,4,33,8,20615,4970,2.133
4,5,33,8,20580,5005,2.136


,fold,accuracy,balanced_accuracy,macro_f1
0,1,0.7448,0.7448,0.7382
1,2,0.6266,0.6279,0.6229
2,3,0.7386,0.7395,0.7372
3,4,0.6483,0.6494,0.6537
4,5,0.6913,0.6931,0.6892


,mean,std
accuracy,0.6899,0.0528
balanced_accuracy,0.6909,0.0524
macro_f1,0.6882,0.0509


,value
Reference macro-F1,0.6907
Resampled macro-F1,0.688232
Difference from reference,-0.002468
Parity threshold,0.67
Parity passed,True


Parity decision: PASS


## 4. Parity findings

The Random Forest trained on features extracted from the 20 Hz representation
achieves:

- mean accuracy: **0.6899 ± 0.0528**;
- mean balanced accuracy: **0.6909 ± 0.0524**;
- mean macro-F1: **0.6882 ± 0.0509**.

The original irregular-sample fused baseline achieved mean macro-F1
**0.6907 ± 0.0480**. The resampled result is lower by only **0.0025**, comfortably
within the predeclared parity requirement of at least 0.67.

| Fold | Original macro-F1 | Resampled macro-F1 | Difference |
|---:|---:|---:|---:|
| 1 | 0.7432 | 0.7382 | -0.0050 |
| 2 | 0.6325 | 0.6229 | -0.0096 |
| 3 | 0.7331 | 0.7372 | +0.0041 |
| 4 | 0.6551 | 0.6537 | -0.0014 |
| 5 | 0.6899 | 0.6892 | -0.0007 |

The result is stable across folds, so the aggregate parity result does not hide
a failed subject group.

The 20 Hz paired-watch artifact is accepted as the shared representation for
subsequent development experiments. This result establishes parity for the
original whole-window statistical features; it does not establish that 20 Hz
is optimal for every high-frequency temporal pattern.

No test data were loaded or evaluated.